In [ ]:
"""
Бот-помічник водія концепт-кара з технологіями Automotive
Відповіді формуються як узагальнення результатів трьох LLM

LLM-1: Qwen/Qwen3-0.6B           (з режимом мислення)
LLM-2: LiquidAI/LFM2.5-1.2B-Instruct  (з 4-bit квантизацією)
LLM-3: HuggingFaceTB/SmolLM2-1.7B-Instruct (легка та швидка)

Встановлення залежностей:
    pip install transformers torch bitsandbytes accelerate
"""

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

# ─────────────────────────────────────────────────────────────
# Ситуації водія (4 задані + 3 на вибір)
# ─────────────────────────────────────────────────────────────
SITUATIONS = {
    1: {
        "name": "Мертва зона дзеркал",
        "prompt": (
            "Автомобіль попереду потрапив у мертву зону бічних дзеркал під час "
            "перестроювання на трасі. Що повинен зробити водій? "
            "Дай 3-4 конкретні практичні дії."
        ),
    },
    2: {
        "name": "Проблеми з тиском шин",
        "prompt": (
            "Датчик TPMS показує критично низький тиск у передній правій шині "
            "під час руху по трасі зі швидкістю 110 км/год. "
            "Які дії водія? Дай 3-4 конкретні кроки."
        ),
    },
    3: {
        "name": "Проблеми з очищенням лобового скла",
        "prompt": (
            "Під час сильного дощу вночі раптово перестали працювати склоочисники. "
            "Видимість різко погіршилась. Що робити водієві? "
            "Дай 3-4 конкретні дії."
        ),
    },
    4: {
        "name": "Аварія (ДТП)",
        "prompt": (
            "Сталась аварія: зіткнення двох автомобілів на перехресті. "
            "Є постраждалі. Опиши покроковий алгоритм дій водія "
            "одразу після ДТП. Дай 4-5 конкретних кроків."
        ),
    },
    # ── три ситуації на вибір ──────────────────────────────────
    5: {
        "name": "Густий туман",
        "prompt": (
            "Водій їде трасою, раптово попадає в зону густого туману, "
            "видимість впала до 20 метрів. Які дії водія? "
            "Дай 3-4 конкретні кроки."
        ),
    },
    6: {
        "name": "Прокол шини на швидкості",
        "prompt": (
            "На швидкості 100 км/год лопнула задня ліва шина. "
            "Авто почало тягнути вбік. Що робити водієві? "
            "Дай 3-4 конкретні термінові дії."
        ),
    },
    7: {
        "name": "Розряд акумулятора",
        "prompt": (
            "Водій намагається завести авто вранці, але акумулятор повністю "
            "розряджений. Поруч є інший автомобіль. "
            "Що потрібно зробити? Дай 3-4 конкретні кроки."
        ),
    },
}


# ─────────────────────────────────────────────────────────────
# LLM-1: Qwen3-0.6B  (паттерн з прикладу лектора №2 і №3)
# ─────────────────────────────────────────────────────────────
class Qwen3Assistant:
    """
    Патерн: приклад лектора №2 — завантаження Qwen3 з enable_thinking.
    Ця модель «думає» перед тим як відповісти (режим <think>).
    """

    MODEL_NAME = "Qwen/Qwen3-0.6B"
    SYSTEM = (
        "Ти — автомобільний асистент з навичками інструктора з безпечного "
        "водіння. Надавай чіткі, практичні поради для водія."
    )

    def __init__(self):
        print(f"[Qwen3] Завантаження {self.MODEL_NAME}...")
        self.tokenizer = AutoTokenizer.from_pretrained(self.MODEL_NAME)
        self.model = AutoModelForCausalLM.from_pretrained(
            self.MODEL_NAME,
            torch_dtype="auto",
            device_map="auto",
        )
        print("[Qwen3] Готово!")

    def ask(self, prompt: str, max_new_tokens: int = 512) -> str:
        messages = [
            {"role": "system", "content": self.SYSTEM},
            {"role": "user", "content": prompt},
        ]
        text = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=True,   # ← режим мислення як у лектора
        )
        model_inputs = self.tokenizer([text], return_tensors="pt").to(
            self.model.device
        )
        generated_ids = self.model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
        )
        output_ids = generated_ids[0][len(model_inputs.input_ids[0]) :].tolist()

        # Відокремлюємо thinking від відповіді (паттерн лектора №2)
        try:
            index = len(output_ids) - output_ids[::-1].index(151668)  # </think>
        except ValueError:
            index = 0

        thinking = self.tokenizer.decode(
            output_ids[:index], skip_special_tokens=True
        ).strip()
        content = self.tokenizer.decode(
            output_ids[index:], skip_special_tokens=True
        ).strip()

        if thinking:
            print(f"  [Qwen3 think] {thinking[:120]}...")
        return content


# ─────────────────────────────────────────────────────────────
# LLM-2: LFM2.5-1.2B-Instruct  (паттерн з прикладу лектора №1)
# ─────────────────────────────────────────────────────────────
class LFM25Assistant:
    """
    Патерн: приклад лектора №1 — BitsAndBytesConfig для 4-bit квантизації.
    Зменшує споживання пам'яті з ~5 ГБ до ~1.5 ГБ.
    """

    MODEL_NAME = "LiquidAI/LFM2.5-1.2B-Instruct"
    SYSTEM = (
        "You are an ADAS (Advanced Driver Assistance System). "
        "Think like an onboard car computer. Give precise, step-by-step "
        "technical safety instructions. Answer in Ukrainian."
    )

    def __init__(self):
        print(f"[LFM2.5] Завантаження {self.MODEL_NAME} (4-bit)...")
        self.tokenizer = AutoTokenizer.from_pretrained(self.MODEL_NAME)

        # ← 4-bit квантизація як у лектора №1
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True,
        )
        self.model = AutoModelForCausalLM.from_pretrained(
            self.MODEL_NAME,
            device_map="auto",
            quantization_config=bnb_config,
        )
        print("[LFM2.5] Готово!")

    def ask(self, prompt: str, max_new_tokens: int = 512) -> str:
        # Формат промту як у лектора №1 (full_prompt з інструкцією)
        full_prompt = (
            f"{self.SYSTEM}\n\n"
            f"Situation:\n{prompt}\n\n"
            f"Step-by-step instructions:\n"
        )
        inputs = self.tokenizer(full_prompt, return_tensors="pt").to(
            self.model.device
        )
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=0.4,   # ← стабільні відповіді як у лектора
                top_p=0.9,
                do_sample=True,
            )
        text = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        # Повертаємо тільки частину після промту (паттерн лектора №1)
        return text[len(full_prompt) :].strip()


# ─────────────────────────────────────────────────────────────
# LLM-3: SmolLM2-1.7B-Instruct  (паттерн з прикладу лектора №3)
# ─────────────────────────────────────────────────────────────
class SmolLM2Assistant:
    """
    Патерн: приклад лектора №3 — QwenChatbot з history.
    Ця модель зберігає контекст розмови між запитами.
    """

    MODEL_NAME = "HuggingFaceTB/SmolLM2-1.7B-Instruct"
    SYSTEM = (
        "Ти — досвідчений автомеханік та інструктор з першої допомоги на дорозі. "
        "Надавай практичні поради з акцентом на безпеку водія та пасажирів."
    )

    def __init__(self):
        print(f"[SmolLM2] Завантаження {self.MODEL_NAME}...")
        self.tokenizer = AutoTokenizer.from_pretrained(self.MODEL_NAME)
        self.model = AutoModelForCausalLM.from_pretrained(
            self.MODEL_NAME,
            torch_dtype="auto",
            device_map="auto",
        )
        self.history = []  # ← зберігаємо контекст як у лектора №3
        print("[SmolLM2] Готово!")

    def ask(self, prompt: str, max_new_tokens: int = 512) -> str:
        # Будуємо повідомлення з системним контекстом (паттерн лектора №3)
        messages = (
            [{"role": "system", "content": self.SYSTEM}]
            + self.history
            + [{"role": "user", "content": prompt}]
        )
        text = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
        inputs = self.tokenizer(text, return_tensors="pt").to(self.model.device)
        response_ids = self.model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
        )[0][len(inputs.input_ids[0]) :].tolist()

        response = self.tokenizer.decode(response_ids, skip_special_tokens=True)

        # Оновлюємо history як у лектора №3
        self.history.append({"role": "user", "content": prompt})
        self.history.append({"role": "assistant", "content": response})
        return response


# ─────────────────────────────────────────────────────────────
# Узагальнення трьох відповідей
# ─────────────────────────────────────────────────────────────
def synthesize_responses(
    situation_name: str,
    qwen_ans: str,
    lfm_ans: str,
    smol_ans: str,
) -> str:
    """
    Об'єднує відповіді трьох моделей в одну фінальну інструкцію.
    Використовує просту логіку злиття без 4-ї моделі
    (щоб не перевантажувати пам'ять).
    """
    separator = "\n" + "─" * 40 + "\n"
    synthesis = (
        f"\n{'═' * 50}\n"
        f"  УЗАГАЛЬНЕНА ВІДПОВІДЬ: {situation_name}\n"
        f"{'═' * 50}\n\n"
        f"[Модель 1 — Qwen3-0.6B / Інструктор з БД]:\n{qwen_ans}\n"
        f"{separator}"
        f"[Модель 2 — LFM2.5-1.2B / Система ADAS]:\n{lfm_ans}\n"
        f"{separator}"
        f"[Модель 3 — SmolLM2-1.7B / Автомеханік]:\n{smol_ans}\n"
        f"{'═' * 50}\n"
    )
    return synthesis


# ─────────────────────────────────────────────────────────────
# Головна функція
# ─────────────────────────────────────────────────────────────
def main():
    print("=" * 50)
    print("  Automotive AI Assistant — завантаження моделей")
    print("=" * 50)

    # Завантажуємо всі три моделі
    qwen = Qwen3Assistant()
    lfm = LFM25Assistant()
    smol = SmolLM2Assistant()

    print("\nВсі три моделі готові!\n")

    while True:
        # Меню ситуацій
        print("\nОберіть ситуацію водія:")
        for num, sit in SITUATIONS.items():
            print(f"  {num}. {sit['name']}")
        print("  0. Вийти")

        choice = input("\nВаш вибір (0-7): ").strip()
        if choice == "0":
            print("До побачення!")
            break
        if not choice.isdigit() or int(choice) not in SITUATIONS:
            print("Невірний вибір, спробуйте ще раз.")
            continue

        sit = SITUATIONS[int(choice)]
        prompt = sit["prompt"]

        print(f"\n{'─' * 50}")
        print(f"Ситуація: {sit['name']}")
        print(f"{'─' * 50}\n")

        # Запитуємо всі три моделі
        print("[1/3] Qwen3 думає...")
        qwen_ans = qwen.ask(prompt)

        print("[2/3] LFM2.5 аналізує...")
        lfm_ans = lfm.ask(prompt)

        print("[3/3] SmolLM2 радить...")
        smol_ans = smol.ask(prompt)

        # Виводимо узагальнення
        result = synthesize_responses(sit["name"], qwen_ans, lfm_ans, smol_ans)
        print(result)

        input("\nНатисніть Enter для продовження...")


if __name__ == "__main__":
    main()

  Automotive AI Assistant — завантаження моделей
[Qwen3] Завантаження Qwen/Qwen3-0.6B...


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.42G [00:00<?, ?B/s]